# Практика · Instruction tuning і RLHF

> Теорія: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

Зошит самодостатній: усе, що тут відбувається, пояснюється на місці, лекцію
відкривати не обовʼязково.

**Задача, яку ми розвʼязуємо.** Навчити маленьку мовну модель **виконувати
вказівку** — і чесно перевірити, чи вміння виконувати вказівку переноситься
на завдання, якого модель не бачила. Потім побудувати **модель винагороди**
на справжніх парах уподобань і знайти точку, у якій оптимізація винагороди
перестає покращувати те, заради чого все робилось.

Що зробимо:

1. дістанемо корпус із твоєї машини — українські переклади повідомлень програм;
2. зберемо **три завдання**, у яких відповідь відома точно, без розмітника;
3. **спершу порахуємо рубежі** — три числа, без яких результат нічого не означає;
4. навчимо моделі з різними сумішами завдань і заміряємо перенесення;
5. подивимось, що модель **насправді** відповідає, а не лише на її точність;
6. навчимо **модель винагороди** за Бредлі-Террі й звіримо свою втрату з бібліотечною;
7. побудуємо криву **best-of-n** і знайдемо, де винагорода росте, а користь уже ні;
8. перевіримо формулу KL для best-of-n прямим підрахунком.

> ⏱ Зошит навчає пʼять невеликих мереж. Заміряно: **близько чотирьох хвилин
> процесорного часу** на чотирьох ядрах без відеокарти. Стінний час на
> завантаженій машині буде більший — це нормально.

In [ ]:
# Фіксуємо кількість потоків ДО імпорту numpy і torch: інакше бібліотеки лінійної
# алгебри розповзаються по всіх ядрах, крутяться в очікуванні, і будь-який замір
# часу перестає щось означати (спотворення буває в десятки разів).
import os
for var in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[var] = '1'

import re, glob, gettext, math, time, random, collections
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
torch.set_num_threads(1)

# Одне зерно на весь зошит: усе, що нижче, відтворюється рядок у рядок.
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Час міряємо ПРОЦЕСОРНИЙ, а не годинником: машина може бути зайнята чужою
# роботою, і годинник тоді показує завантаження, а не наші обчислення.
STARTED = time.process_time()

print('torch', torch.__version__, '· потоків', torch.get_num_threads())
print('зерно', SEED)

## 1 · Звідки беремо дані

Курс не тримає текстів у репозиторії. Корпус ми беремо **з твоєї машини**:
це файли перекладів `.mo`, які лежать у `/usr/share/locale/uk/LC_MESSAGES/`.
Кожна встановлена програма кладе туди свій файл, а в ньому — пари
«англійський оригінал → український переклад».

Цей корпус має рідкісну властивість: у кожного рядка є **три речі, які ми
знаємо точно, без жодної розмітки**.

* з якої він **програми** — це ім'я файлу;
* який його **англійський оригінал** — він лежить поруч у тому самому файлі;
* який його **текст** — власне переклад.

Саме з цих трьох речей ми зараз зробимо три завдання з чесними відповідями.

⚠️ Числа в тебе будуть інші, ніж тут: набір встановлених програм у кожного свій.
Відтворюється не кількість рядків, а **форма** результатів — з якого боку від
рубежу стоять числа й у який бік ідуть криві.

In [ ]:
WORD = re.compile(r"[а-яїієґ]+(?:[\'ʼ’][а-яїієґ]+)*")

def words(text):
    """Розбиває український текст на слова. Латиниця й цифри нас тут не цікавлять:
    ми вчимо модель на перекладі, а не на іменах файлів усередині повідомлення."""
    return WORD.findall(text.lower())

def load_messages(min_chars=20):
    """Читає всі українські каталоги перекладів, які є на машині.

    Повертає список трійок (програма, англійський оригінал, український переклад).
    Службовий запис із метаданими каталогу (він містить 'Project-Id') відкидаємо."""
    rows = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        program = os.path.basename(path)[:-3]
        try:
            with open(path, 'rb') as f:
                catalog = gettext.GNUTranslations(f)._catalog
        except Exception:
            continue                      # пошкоджений або незвичний файл просто пропускаємо
        for source, translation in catalog.items():
            if not isinstance(source, str) or not isinstance(translation, str):
                continue
            if len(translation) <= min_chars or 'Project-Id' in translation:
                continue
            rows.append((program, source, translation))
    return rows

raw_rows = load_messages()
print(f'сирих повідомлень: {len(raw_rows)}')
print(f'програм: {len(set(p for p, _, _ in raw_rows))}')

### Фільтр і перемішування

Дуже короткі рядки («Гаразд») і дуже довгі (цілі абзаци довідки) нам заважають:
з перших нема чого дописувати, другі не влазять у контекст маленької моделі.
Лишаємо рядки від 6 до 20 слів.

Перемішуємо **один раз із фіксованим зерном** — і далі всі поділи робимо
зрізами перемішаного списку. Так навчальна й перевірна частини не залежать
від того, у якому порядку операційна система віддала файли.

In [ ]:
rows = [(program, source, words(translation)) for program, source, translation in raw_rows]
rows = [(p, s, w) for p, s, w in rows if 6 <= len(w) <= 20]
random.Random(0).shuffle(rows)

# Беремо перші 12 000 рядків уже ПЕРЕМІШАНОГО списку — це випадкова підвибірка,
# а не префікс вихідних даних. Менший обсяг тримає зошит у межах кількох хвилин.
rows = rows[:12000]

lengths = [len(w) for _, _, w in rows]
print(f'рядків після фільтра: {len(rows)}')
print(f'довжина: середня {sum(lengths)/len(lengths):.2f} слова · медіана {sorted(lengths)[len(lengths)//2]}')

### Перш ніж рахувати — подивись на дані очима

Найдешевша перевірка в усьому зошиті. Вона регулярно рятує від помилки,
яку жодна метрика не показує.

In [ ]:
for index in (0, 1, 2):
    program, source, translation = rows[index]
    print(f'програма : {program}')
    print(f'оригінал : {source[:80]}')
    print(f'переклад : {" ".join(translation)[:80]}')
    print()

## 2 · Три завдання, у яких відповідь відома точно

Тепер зробимо з корпусу **набір для донавчання на вказівках**. Кожен приклад
має три частини: вказівка (що зробити), вхід (над чим), відповідь (що видати).

| завдання | вказівка каже | звідки відома відповідь |
|---|---|---|
| `клас` | це повідомлення про збій чи про успіх? | з англійського оригіналу того самого рядка |
| `далі` | допиши продовження рядка | друга половина самого рядка |
| `звідки` | з якої програми цей рядок? | з імені файлу перекладу |

Мітку «збій / успіх» здобуваємо **правилом** по англійському оригіналу: якщо
в ньому є `error`, `fail`, `cannot` і подібні — це збій. Правило просте, і за
хвилину ми чесно поміряємо, наскільки воно діряве.

Завдання `звідки` обмежуємо вісьмома найбільшими програмами: у корпусі їх
сотні, і більшість дає по кілька рядків — на такому не вчиться ніхто.

In [ ]:
FAIL = re.compile(r"\b(error|fail|failed|cannot|unable|invalid|denied|corrupt)\b", re.I)

programs = collections.Counter(program for program, _, _ in rows)
TOP_PROGRAMS = [p for p, _ in programs.most_common(8)]

print('вісім найбільших програм:')
for p in TOP_PROGRAMS:
    print(f'  {p:<16} {programs[p]:>5} рядків')
print(f'\nразом у них {sum(programs[p] for p in TOP_PROGRAMS)} рядків '
      f'із {len(rows)} ({sum(programs[p] for p in TOP_PROGRAMS)/len(rows):.1%})')

### Наскільки діряве правило «збій / успіх»

Правило ловить `cannot`, але не ловить `could not`, `is not`, `no such`,
`timed out` і ще з десяток зворотів заперечення. Це означає, що частина
повідомлень про збій дістає мітку «успіх» — і помилки всі **в один бік**.

Порахуймо це число. Воно потрібне не для краси: воно означає, що
**стеля цього завдання нижча за одиницю не через модель**, і модель, яка
правильно зрозуміла «could not be loaded», за це буде покарана.

In [ ]:
# Звороти заперечення, яких правило НЕ ловить. Це верхня оцінка дірки:
# «сервер цього не потребує» теж сюди потрапить, хоч і не є збоєм.
MISSED = re.compile(r"\b(could not|couldn't|is not|are not|does not|doesn't|"
                    r"no such|not found|missing|refused|timed out|aborted|"
                    r"rejected|unsupported)\b", re.I)

labels = collections.Counter('збій' if FAIL.search(s) else 'успіх' for _, s, _ in rows)
hole = [s for _, s, _ in rows if not FAIL.search(s) and MISSED.search(s)]

print(f'збій  : {labels["збій"]:>6}  ({labels["збій"]/len(rows):.2%})')
print(f'успіх : {labels["успіх"]:>6}  ({labels["успіх"]/len(rows):.2%})')
print(f'\nз них містять незловлене заперечення: {len(hole)} '
      f'({len(hole)/labels["успіх"]:.2%} класу «успіх»)')
print('\nприклади того, що правило проґавило:')
for s in hole[:3]:
    print('  ·', s.replace('\n', ' ')[:72])

## 3 · Словник і службові токени

Модель працює з числами, а не зі словами, тож треба словник. До звичайних слів
додаємо **службові токени**, яких у жодному тексті не буває:

* `<клас>`, `<далі>`, `<звідки>` — вказівки;
* `<sep>` — роздільник між входом і відповіддю;
* `<збій>`, `<успіх>` — мітки класу;
* по одному токену на кожну з восьми програм.

Чому окремі токени, а не звичайні слова? Бо звичайне слово може трапитись
усередині повідомлення — і межа між частинами поїде. Службовий токен такої
проблеми не має за побудовою.

⚠️ Зверни увагу на ціну цього рішення: **ембединги службових токенів
доводиться вчити з нуля**, бо в текстах їх не було. Через дві клітинки це
виявиться найважливішою обставиною всього зошита.

In [ ]:
split_at = int(0.8 * len(rows))
train_rows, test_rows = rows[:split_at], rows[split_at:]

counts = collections.Counter(w for _, _, ws in train_rows for w in ws)

SPECIAL = (['<pad>', '<bos>', '<eos>', '<unk>',
            '<клас>', '<далі>', '<звідки>', '<sep>', '<збій>', '<успіх>']
           + [f'<{p}>' for p in TOP_PROGRAMS])
itos = SPECIAL + [w for w, c in counts.most_common() if c >= 5]
stoi = {w: i for i, w in enumerate(itos)}
PAD, BOS, EOS, UNK = 0, 1, 2, 3
VOCAB = len(itos)
MAXLEN = 32

def ids(word_list):
    """Слова -> номери. Невідоме слово стає <unk>."""
    return [stoi.get(w, UNK) for w in word_list]

# Множина допустимих відповідей кожного завдання. Знадобиться двічі:
# для випадкового рубежу (1/K) і для «обмеженого» способу оцінювання.
ANSWERS = {'клас':   ids(['<збій>', '<успіх>']),
           'звідки': ids([f'<{p}>' for p in TOP_PROGRAMS]),
           'далі':   list(range(len(SPECIAL), VOCAB))}

print(f'навчальних рядків {len(train_rows)} · перевірних {len(test_rows)}')
print(f'словник {VOCAB} = {len(SPECIAL)} службових + {VOCAB - len(SPECIAL)} слів')
for task, allowed in ANSWERS.items():
    print(f'  допустимих відповідей у «{task}»: {len(allowed)}')

## 4 · Один приклад як послідовність токенів

Формат один для всіх трьох завдань:

```
<bos> <вказівка> вхід… <sep> відповідь… <eos>
```

Функція нижче повертає **два** значення: саму послідовність і те, **скільки
останніх токенів є відповіддю**. Друге число знадобиться, щоб рахувати помилку
лише на відповіді.

In [ ]:
def make_example(task, program, source, translation):
    """Один приклад у форматі «вказівка → вхід → відповідь».

    Повертає (послідовність, скільки токенів у кінці є відповіддю).
    Для завдання «звідки» повертає None, якщо програма не входить у вісімку."""
    if task == 'клас':
        answer = ['<збій>'] if FAIL.search(source) else ['<успіх>']
        given = translation
    elif task == 'далі':
        half = len(translation) // 2
        given, answer = translation[:half], translation[half:]
    else:
        if program not in TOP_PROGRAMS:
            return None
        answer, given = [f'<{program}>'], translation
    sequence = ([BOS] + ids([f'<{task}>']) + ids(given)[:12]
                + ids(['<sep>']) + ids(answer)[:6] + [EOS])
    # +1 — це <eos>: закінчувати відповідь модель теж мусить навчитись
    return sequence, len(ids(answer)[:6]) + 1

program, source, translation = rows[3]
print('вихідний рядок:', ' '.join(translation))
print()
for task in ('клас', 'далі', 'звідки'):
    made = make_example(task, program, source, translation)
    if made is None:
        print(f'{task:<8}: пропущено (програма {program} не входить у вісімку)')
        continue
    sequence, answer_len = made
    shown = [itos[t] for t in sequence]
    border = len(sequence) - answer_len
    print(f'{task:<8}: {" ".join(shown[:border])}   ||   {" ".join(shown[border:])}')
print('\nліворуч від || — дано моделі; праворуч — те, що вона мусить породити')

## 5 · Рубежі — до того, як зʼявиться модель

Це найважливіша клітинка зошита, і вона навмисне стоїть **перед** навчанням.

Точність без рубежу — не результат, а цифра. Щоб число щось означало, поруч
мусять стояти три:

1. **найчастіша відповідь** — константа, узята з навчальної частини (не з
   перевірної: інакше це підглядання у відповідь);
2. **випадкова відповідь** — точне сподівання `1/K`, де `K` — скільки
   допустимих відповідей. Жеребкувати не треба, число рахується точно;
3. **стеля** — модель, навчена лише на цьому завданні. Її ми дістанемо пізніше,
   бо вона потребує навчання.

Перші два рахуємо тим самим кодом, що й точність моделі: підставляємо замість
передбачення константу й проганяємо через ту саму функцію. Формула «на папері»
й код розходяться частіше, ніж здається, і правий завжди код.

In [ ]:
EVAL_SIZE = 400

def gold_answer(task, program, source, translation):
    """Перший токен правильної відповіді — те саме, що ми будемо порівнювати."""
    made = make_example(task, program, source, translation)
    if made is None:
        return None
    sequence, answer_len = made
    return sequence[len(sequence) - answer_len]

# Рядки, на яких оцінюємо. ОДНІ Й ТІ САМІ для всіх систем і для всіх рубежів.
eval_rows = {}
for task in ('клас', 'далі', 'звідки'):
    chosen = []
    for program, source, translation in test_rows:
        gold = gold_answer(task, program, source, translation)
        if gold is None:
            continue
        # «далі» з правильною відповіддю <unk> нікому не по зубах: жодна система
        # не має права відповісти «невідоме слово», тож такі рядки прибираємо
        if task == 'далі' and gold == UNK:
            continue
        chosen.append((program, source, translation))
        if len(chosen) >= EVAL_SIZE:
            break
    eval_rows[task] = chosen
    print(f'оцінюємо «{task}» на {len(chosen)} рядках')

In [ ]:
floors = {}
for task in ('клас', 'далі', 'звідки'):
    # найчастіша відповідь беремо з НАВЧАЛЬНОЇ частини
    seen = collections.Counter()
    for program, source, translation in train_rows:
        gold = gold_answer(task, program, source, translation)
        if gold is not None and gold != UNK:
            seen[gold] += 1
    constant = seen.most_common(1)[0][0]
    # а міряємо на перевірній — тим самим порівнянням, що й для моделі
    hits = sum(1 for p, s, t in eval_rows[task] if gold_answer(task, p, s, t) == constant)
    floors[task] = {'найчастіша': hits / len(eval_rows[task]),
                    'випадкова': 1.0 / len(ANSWERS[task]),
                    'константа': itos[constant]}

print(f'{"завдання":<10}{"найчастіша":>12}{"відповідь":>14}{"випадкова":>12}{"варіантів":>12}')
for task, f in floors.items():
    print(f'{task:<10}{f["найчастіша"]:>12.4f}{f["константа"]:>14}'
          f'{f["випадкова"]:>12.4f}{len(ANSWERS[task]):>12}')

Подивись на таблицю уважно — вона пояснює, чому одного рубежу мало.

У завданні `клас` головна планка — **константа**: класи дуже нерівні, і мовчазне
«завжди успіх» уже дає багато. У завданні `звідки` константа слабша, але
випадковий вибір із восьми дає лише 0.125. А в `далі` обидва рубежі мізерні,
бо допустимих відповідей тисячі.

Одне й те саме число точності в цих трьох завданнях означає провал, посередність
і чудовий результат.

## 6 · Модель

Той самий трансформер, що в попередніх темах курсу, лише маленький: розмірність
96, два шари, чотири голови уваги. Ми додали один метод — `hidden`, який
повертає стани **до** проєкції у словник. Він потрібен для швидкості: проєкція
з 96 вимірів у весь словник коштує більше, ніж усе решта, а рахувати її треба
лише в тих позиціях, де стоїть помилка.

In [ ]:
def pad_batch(sequences):
    """Доповнює послідовності до однакової довжини нулями (<pad>)."""
    width = max(len(s) for s in sequences)
    return torch.tensor([s + [PAD] * (width - len(s)) for s in sequences])

class TinyLM(nn.Module):
    def __init__(self, vocab, d=96, layers=2, heads=4):
        super().__init__()
        self.emb = nn.Embedding(vocab, d, padding_idx=PAD)
        self.pos = nn.Embedding(MAXLEN, d)
        layer = nn.TransformerEncoderLayer(d, heads, dim_feedforward=4 * d,
                                           batch_first=True, dropout=0.0, norm_first=True)
        self.body = nn.TransformerEncoder(layer, layers)
        self.out = nn.Linear(d, vocab)

    def hidden(self, x):
        """Стани перед проєкцією у словник. Маска верхнього трикутника не дає
        позиції зазирнути вперед — інакше завдання «допиши далі» стало б
        тривіальним."""
        length = x.size(1)
        h = self.emb(x) + self.pos(torch.arange(length))
        mask = torch.triu(torch.full((length, length), float('-inf')), 1)
        return self.body(h, mask=mask, src_key_padding_mask=(x == PAD))

    def forward(self, x):
        return self.out(self.hidden(x))

probe = TinyLM(VOCAB)
print('параметрів:', sum(p.numel() for p in probe.parameters()))
print('з них у проєкції в словник:', sum(p.numel() for p in probe.out.parameters()))

## 7 · Навчання: помилку рахуємо тільки на відповіді

Ось деталь, яка вирішує все. Звичайну мовну модель учать передбачати **кожен**
наступний токен. При донавчанні на вказівках так робити не треба: токени
вказівки й входу пише користувач, угадувати їх ні до чого.

У коді нижче ми збираємо список позицій, які **справді** несуть помилку —
це рівно позиції відповіді, — і рахуємо перехресну ентропію лише в них.

In [ ]:
STEPS, BATCH, LR = 700, 64, 1e-2

def train_instruct(data, seed=0, steps=STEPS):
    """Донавчання на вказівках. data — список пар (послідовність, довжина відповіді)."""
    torch.manual_seed(seed)
    model = TinyLM(VOCAB)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps, pct_start=0.1)
    picker = random.Random(seed)
    index = list(range(len(data)))
    started = time.process_time()
    for _ in range(steps):
        batch = [data[j] for j in picker.sample(index, BATCH)]
        x = pad_batch([s for s, _ in batch])
        # позиції, у яких стоїть помилка: рівно токени відповіді
        at_row, at_col, target = [], [], []
        for i, (sequence, answer_len) in enumerate(batch):
            for j in range(len(sequence) - 1 - answer_len, len(sequence) - 1):
                at_row.append(i); at_col.append(j); target.append(sequence[j + 1])
        states = model.hidden(x[:, :-1])
        chosen = states[torch.tensor(at_row), torch.tensor(at_col)]
        loss = F.cross_entropy(model.out(chosen), torch.tensor(target))
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sch.step()
    return model, time.process_time() - started

def build_dataset(tasks, source_rows):
    """Суміш завдань: кожен рядок перетворюємо на приклад для кожного завдання."""
    out = []
    for program, source, translation in source_rows:
        for task in tasks:
            made = make_example(task, program, source, translation)
            if made is not None:
                out.append(made)
    return out

for tasks in (['клас', 'далі'], ['клас', 'далі', 'звідки'], ['звідки']):
    print(f'{"+".join(tasks):<22} прикладів {len(build_dataset(tasks, train_rows))}')

## 8 · Як оцінюємо: два способи, і різниця між ними важлива

Модель бачить усе до роздільника `<sep>` і мусить назвати перший токен
відповіді. Порівнюємо його з правильним. Але «назвати» можна двома способами:

* **вільний вибір** — argmax по **всьому** словнику. Модель може відповісти
  будь-чим, зокрема тим, що взагалі не є допустимою відповіддю;
* **обмежений вибір** — argmax лише серед допустимих відповідей завдання.
  Модель ніби питають «яка з восьми програм?» замість «скажи що-небудь».

Різниця здається технічною. Зараз вона виявиться вирішальною.

In [ ]:
@torch.no_grad()
def accuracy(model, task):
    """Повертає (вільна точність, обмежена точність) на однакових рядках."""
    model.eval()
    allowed = torch.tensor(ANSWERS[task])
    contexts, golds = [], []
    for program, source, translation in eval_rows[task]:
        sequence, answer_len = make_example(task, program, source, translation)
        border = len(sequence) - 1 - answer_len          # позиція <sep>
        contexts.append(sequence[:border + 1])
        golds.append(sequence[border + 1])
    free = restricted = 0
    for start in range(0, len(contexts), 64):
        chunk = contexts[start:start + 64]
        states = model.hidden(pad_batch(chunk))
        last = states[torch.arange(len(chunk)),
                      torch.tensor([len(c) - 1 for c in chunk])]
        logits = model.out(last)
        gold = torch.tensor(golds[start:start + 64])
        free += int((logits.argmax(1) == gold).sum())
        restricted += int((allowed[logits[:, allowed].argmax(1)] == gold).sum())
    model.train()
    return free / len(contexts), restricted / len(contexts)

print('функція готова · оцінюватимемо на', {t: len(r) for t, r in eval_rows.items()})

## 9 · Перший замір: ховаємо завдання «звідки»

Навчаємо на суміші `клас + далі`. Завдання `звідки` модель не бачила жодного
разу — ані вказівки, ані відповідей. Питаємо саме його.

Це і є **нульовий постріл**: перевірка того, чи вміння виконувати вказівку
переноситься на нове завдання.

In [ ]:
model_without, seconds = train_instruct(build_dataset(['клас', 'далі'], train_rows))
print(f'навчено за {seconds:.0f} с процесорних\n')
zero_shot = {}
for task in ('клас', 'далі', 'звідки'):
    free, restricted = accuracy(model_without, task)
    zero_shot[task] = (free, restricted)
    print(f'{task:<8} вільно {free:.4f} · обмежено {restricted:.4f}')

Зупинись на рядку `звідки`. Вільна точність — **рівно нуль**, і це не збіг.

Модель ніколи не бачила токенів `<gcc>`, `<libvirt>` і решти шести: вони стоять
у відповідях **лише** завдання `звідки`. Під час навчання softmax тиснув їхні
логіти вниз як негативні приклади. Тому вільний argmax **фізично не може** їх
обрати, і нуль на виході — це властивість розмітки, а не факт про перенесення.

Саме тому потрібен обмежений вибір: він ставить модель у становище, у якому
відповісти правильно взагалі можливо. Порівняймо це число з рубежами.

In [ ]:
free, restricted = zero_shot['звідки']
print(f'нульовий постріл, обмежений вибір : {restricted:.4f}')
print(f'рубіж «випадкова відповідь»       : {floors["звідки"]["випадкова"]:.4f}')
print(f'рубіж «найчастіша відповідь»      : {floors["звідки"]["найчастіша"]:.4f}')
print()
if restricted < floors['звідки']['випадкова']:
    print('Нульовий постріл НИЖЧИЙ навіть за випадковий вибір.')
    print('Це не «трохи не вийшло» — це гірше за монетку з вісьмома боками.')
else:
    print('Нульовий постріл вище за випадковий вибір — щось таки перенеслось.')

## 10 · Стеля: а чи вчиться це завдання взагалі?

Попереднє число саме по собі нічого не доводить. Низька точність могла означати,
що перенесення немає, — а могла означати, що наш стенд заслабкий і завдання
`звідки` на ньому не вчиться в принципі.

Розрізняє ці дві причини **стеля**: та сама модель, той самий бюджет навчання,
але навчена **лише** на завданні `звідки`.

In [ ]:
model_ceiling, seconds = train_instruct(build_dataset(['звідки'], train_rows))
ceiling_free, ceiling_restricted = accuracy(model_ceiling, 'звідки')
print(f'стеля («тільки звідки»), {seconds:.0f} с : {ceiling_restricted:.4f}')
print(f'рубіж «найчастіша»                 : {floors["звідки"]["найчастіша"]:.4f}')
print(f'запас стелі над рубежем            : {ceiling_restricted - floors["звідки"]["найчастіша"]:+.4f}')

Стеля впевнено вища за рубіж — отже завдання **вчиться**, і стенд не заслабкий.
Тепер нульовий постріл справді щось означає: розрив між ним і стелею є
вимірюванням того, чого бракує.

Додамо третю систему для повноти: модель, навчена на **всіх трьох** завданнях.
Вона бачила `звідки` — тобто це не перенесення, а звичайне навчання в суміші.

In [ ]:
model_all, seconds = train_instruct(build_dataset(['клас', 'далі', 'звідки'], train_rows))
all_free, all_restricted = accuracy(model_all, 'звідки')
print(f'«усі три», {seconds:.0f} с : {all_restricted:.4f}\n')

print(f'{"система":<32}{"звідки":>10}')
print(f'{"випадкова відповідь":<32}{floors["звідки"]["випадкова"]:>10.4f}')
print(f'{"найчастіша відповідь":<32}{floors["звідки"]["найчастіша"]:>10.4f}')
print(f'{"нульовий постріл (не бачила)":<32}{zero_shot["звідки"][1]:>10.4f}')
print(f'{"бачила в суміші з двома іншими":<32}{all_restricted:>10.4f}')
print(f'{"стеля (тільки це завдання)":<32}{ceiling_restricted:>10.4f}')

## 11 · Що модель насправді відповідає

Точність — це одне число, і воно ховає причину. Подивимось на самі
передбачення: які чотири токени модель вважає найімовірнішими після `<sep>`,
коли її просять назвати програму.

In [ ]:
@torch.no_grad()
def top_guesses(model, task, how_many=4, examples=4):
    model.eval()
    out = []
    for program, source, translation in eval_rows[task][:examples]:
        sequence, answer_len = make_example(task, program, source, translation)
        border = len(sequence) - 1 - answer_len
        logits = model(torch.tensor([sequence[:border + 1]]))[0, -1]
        best = [itos[i] for i in logits.topk(how_many).indices.tolist()]
        out.append((' '.join(translation)[:40], itos[sequence[border + 1]], best))
    model.train()
    return out

print('МОДЕЛЬ, ЯКА НЕ БАЧИЛА ЗАВДАННЯ «ЗВІДКИ»:')
for text, gold, best in top_guesses(model_without, 'звідки'):
    print(f'  вхід   : {text}')
    print(f'  еталон : {gold}   модель: {", ".join(best)}')
print()
print('МОДЕЛЬ, НАВЧЕНА ЛИШЕ НА «ЗВІДКИ» (стеля):')
for text, gold, best in top_guesses(model_ceiling, 'звідки'):
    print(f'  вхід   : {text}')
    print(f'  еталон : {gold}   модель: {", ".join(best)}')

Тепер видно причину, а не лише число. Модель, яка не бачила завдання, після
`<sep>` пропонує те, що вона звикла там бачити на **інших** завданнях, —
мітки класу або звичайні слова. Вона не «не зрозуміла питання»: вона
відповідає в тому єдиному форматі, який уміє.

**Висновок першої половини зошита.** Переноситься не абстрактна слухняність,
а вміння породжувати відповідь **тієї форми**, яку модель уже породжувала.
Нова форма відповіді не переноситься ніяк.

## 12 · Друга половина: винагорода

Донавчання на вказівках нічого не каже про те, **яка з двох правильних
відповідей краща**. Для цього потрібні уподобання.

Живих розмітників у нас немає, тож ми беремо уподобання, які є в даних
безкоштовно й чесно: **справжній переклад кращий за те, що породила модель**.
Це не вигадка — обидва тексти справжні, і порядок між ними не викликає сумнівів.

Спершу треба сама модель, яка породжує. Навчимо звичайну мовну модель на тих
самих рядках — без жодних вказівок, просто продовжувати текст. Далі зватимемо
її **політикою**: слово підкреслює, що нас цікавить не будова мережі, а те,
що вона робить.

In [ ]:
PROMPT_LEN, NEW_TOKENS, N_MAX = 4, 10, 16

def encode(word_list):
    return [BOS] + ids(word_list) + [EOS]

policy_rows = [ws for _, _, ws in train_rows if len(ws) >= PROMPT_LEN + 4]
policy_test = [ws for _, _, ws in test_rows if len(ws) >= PROMPT_LEN + 4][:120]

torch.manual_seed(0)
policy = TinyLM(VOCAB)
opt = torch.optim.AdamW(policy.parameters(), lr=LR, weight_decay=0.01)
sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=STEPS, pct_start=0.1)
data = [encode(ws) for ws in policy_rows]
picker = random.Random(0)
index = list(range(len(data)))
started = time.process_time()
for _ in range(STEPS):
    x = pad_batch([data[j] for j in picker.sample(index, BATCH)])
    # звичайна мовна модель: помилка на ВСІХ позиціях, бо тут немає «даного» і «шуканого»
    loss = F.cross_entropy(policy(x[:, :-1]).reshape(-1, VOCAB),
                           x[:, 1:].reshape(-1), ignore_index=PAD)
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
    opt.step(); sch.step()
policy.eval()
print(f'політика навчена за {time.process_time() - started:.0f} с · '
      f'рядків {len(policy_rows)} · перевірних {len(policy_test)}')

In [ ]:
@torch.no_grad()
def sample_continuations(prompt, how_many, seed):
    """Тягне how_many продовжень підказки однією пачкою.

    Жеребкуємо з розподілу (а не беремо найімовірніше), бо нам потрібні РІЗНІ
    кандидати: з однакових вибирати нема чого."""
    gen = torch.Generator().manual_seed(seed)
    sequences = [list(prompt) for _ in range(how_many)]
    finished = [False] * how_many
    for _ in range(NEW_TOKENS):
        if all(finished):
            break
        logits = policy(pad_batch([s[-MAXLEN:] for s in sequences]))
        last = torch.stack([logits[i, len(sequences[i]) - 1] for i in range(how_many)])
        drawn = torch.multinomial(torch.softmax(last, -1), 1, generator=gen).squeeze(1)
        for i in range(how_many):
            if finished[i]:
                continue
            token = int(drawn[i])
            sequences[i].append(token)
            if token == EOS:
                finished[i] = True
    return [s[len(prompt):] for s in sequences]

example_ids = encode(policy_test[0])
prompt, reference = example_ids[:PROMPT_LEN], example_ids[PROMPT_LEN:]
print('підказка :', ' '.join(itos[t] for t in prompt))
print('еталон   :', ' '.join(itos[t] for t in reference))
for k, cand in enumerate(sample_continuations(prompt, 3, 777)):
    print(f'кандидат {k}:', ' '.join(itos[t] for t in cand))

## 13 · Пари уподобань

Для кожного з 600 навчальних рядків беремо перші три слова як підказку.
Краща відповідь — **справжнє** продовження рядка. Гірша — те, що породила
політика. Пара готова.

In [ ]:
started = time.process_time()
pairs = []
for i, ws in enumerate(policy_rows[:600]):
    full = encode(ws)
    prompt, reference = full[:PROMPT_LEN], full[PROMPT_LEN:]
    generated = sample_continuations(prompt, 1, 1000 + i)[0]
    pairs.append((prompt + reference, prompt + generated))

identical = sum(1 for better, worse in pairs if better == worse)
print(f'пар {len(pairs)} · {time.process_time() - started:.0f} с')
print(f'пар, де породжене дослівно збіглося з еталоном: {identical} '
      f'({identical/len(pairs):.2%})')
print('такі пари нічого не вчать: правильної відповіді в них не існує')

## 14 · Бредлі-Террі: наша втрата проти бібліотечної

Модель винагороди дає тексту одне число. Даних із числами в нас немає — є лише
порядок. Перетворює порядок на число модель **Бредлі-Террі**: ймовірність, що
людина обере A замість B, дорівнює σ(r(A) − r(B)), де σ — логістична функція.

Втрата — мінус логарифм цієї ймовірності:

```
втрата = −ln σ( r(краще) − r(гірше) )
```

Перш ніж будувати мережу, переконаймось, що ми розуміємо формулу правильно.
Вона мусить збігтися з бібліотечною втратою `binary_cross_entropy_with_logits`,
якщо подати їй різницю оцінок і мітку «одиниця» (тобто «перше справді краще»).

In [ ]:
difference = torch.tensor([2.5, 0.0, -1.3, 4.0, -0.2])       # r(краще) − r(гірше)

ours = -F.logsigmoid(difference)                              # наша формула
library = F.binary_cross_entropy_with_logits(
    difference, torch.ones_like(difference), reduction='none')

print('різниця оцінок :', [round(float(v), 4) for v in difference])
print('наша втрата    :', [round(float(v), 4) for v in ours])
print('бібліотечна    :', [round(float(v), 4) for v in library])
assert torch.allclose(ours, library, atol=1e-6), 'розрахунок розійшовся!'
print('\n✅ збігається')
print(f'при нульовій різниці втрата = ln 2 = {math.log(2):.4f} — модель кидає монетку')

In [ ]:
class Reward(nn.Module):
    """Оцінює пару «підказка + продовження» одним числом.

    Тут маска верхнього трикутника НЕ потрібна: ми не породжуємо текст, а
    оцінюємо готовий, тож кожна позиція може дивитись на весь текст."""
    def __init__(self, vocab, d=64):
        super().__init__()
        self.emb = nn.Embedding(vocab, d, padding_idx=PAD)
        self.pos = nn.Embedding(MAXLEN, d)
        layer = nn.TransformerEncoderLayer(d, 4, dim_feedforward=4 * d,
                                           batch_first=True, dropout=0.0, norm_first=True)
        self.body = nn.TransformerEncoder(layer, 2)
        self.head = nn.Linear(d, 1)

    def forward(self, x):
        h = self.emb(x) + self.pos(torch.arange(x.size(1)))
        h = self.body(h, src_key_padding_mask=(x == PAD))
        keep = (x != PAD).float().unsqueeze(-1)
        # усереднюємо по реальних токенах, доповнення в середнє не входить
        return self.head((h * keep).sum(1) / keep.sum(1).clamp(min=1)).squeeze(-1)

torch.manual_seed(0)
reward_model = Reward(VOCAB)
opt = torch.optim.AdamW(reward_model.parameters(), lr=3e-3, weight_decay=0.01)
picker = random.Random(0)
index = list(range(len(pairs)))
started = time.process_time()
for _ in range(400):
    batch = picker.sample(index, 32)
    better = pad_batch([pairs[j][0][:MAXLEN] for j in batch])
    worse = pad_batch([pairs[j][1][:MAXLEN] for j in batch])
    loss = -F.logsigmoid(reward_model(better) - reward_model(worse)).mean()
    opt.zero_grad(); loss.backward(); opt.step()
reward_model.eval()
print(f'модель винагороди навчена за {time.process_time() - started:.0f} с · '
      f'остання втрата {float(loss):.4f}')

### Чи вгадує вона на невидимих парах

Рубіж тут очевидний і дорівнює **0.5**: модель, яка кидає монетку, вгадує
половину. Беремо перевірні рядки, яких у навчанні не було, будуємо з них такі
самі пари й дивимось, чи ставить модель еталону вищу оцінку.

In [ ]:
agreed = 0
with torch.no_grad():
    for i, ws in enumerate(policy_test[:100]):
        full = encode(ws)
        prompt, reference = full[:PROMPT_LEN], full[PROMPT_LEN:]
        generated = sample_continuations(prompt, 1, 9000 + i)[0]
        better = float(reward_model(pad_batch([(prompt + reference)[:MAXLEN]])))
        worse = float(reward_model(pad_batch([(prompt + generated)[:MAXLEN]])))
        agreed += int(better > worse)
print(f'згода на 100 невидимих парах : {agreed/100:.4f}')
print(f'рубіж «монетка»              : 0.5000')

## 15 · Best-of-n: де винагорода перестає означати користь

Найпростіший спосіб оптимізувати винагороду — породити `n` кандидатів і взяти
того, кому модель винагороди дала найбільше. Модель при цьому не міняється
взагалі; міняється лише те, як ми нею користуємось.

Кандидатів тягнемо **один раз** — по 16 на підказку, — а best-of-n беремо серед
**перших** `n`. Так набори вкладені один в одного, і різниця між сусідніми `n` є
справді ефектом `n`, а не новим жеребкуванням.

Міряємо чотири величини:

* **винагорода** обраного — те, що ми оптимізуємо;
* **справжня метрика** — перекриття токенів з еталонним продовженням. Це
  груба міра, але вона незалежна від моделі винагороди, і саме тому потрібна;
* **оракул** — найкраща справжня метрика серед тих самих `n` кандидатів.
  Показує, чи є з чого вибирати взагалі;
* **KL** — наскільки відбір відсунув нас від початкової політики. Для
  best-of-n є закрита формула, її перевіримо в наступній клітинці.

In [ ]:
def token_f1(generated, reference):
    """Справжня метрика: наскільки збіглися токени. Незалежна від винагороди."""
    a = collections.Counter(t for t in generated if t != EOS)
    b = collections.Counter(t for t in reference if t != EOS)
    overlap = sum((a & b).values())
    if not overlap:
        return 0.0
    return 2 * overlap / (sum(a.values()) + sum(b.values()))

started = time.process_time()
scored = []
with torch.no_grad():
    for i, ws in enumerate(policy_test[:100]):
        full = encode(ws)
        prompt, reference = full[:PROMPT_LEN], full[PROMPT_LEN:]
        candidates = sample_continuations(prompt, N_MAX, 4000 + i)
        rewards = reward_model(pad_batch([(prompt + c)[:MAXLEN] for c in candidates])).tolist()
        truths = [token_f1(c, reference) for c in candidates]
        reference_reward = float(reward_model(pad_batch([(prompt + reference)[:MAXLEN]])))
        scored.append((rewards, truths, reference_reward))
print(f'зібрано {len(scored)} підказок по {N_MAX} кандидатів · '
      f'{time.process_time() - started:.0f} с')

In [ ]:
print(f'{"n":>3}{"винагорода":>13}{"справжня":>11}{"оракул":>9}'
      f'{"переганяє еталон":>19}{"KL, нат":>10}')
curve = {}
for n in (1, 2, 4, 8, 16):
    picked_reward, picked_truth, oracle, beats = [], [], [], []
    for rewards, truths, reference_reward in scored:
        best = max(range(n), key=lambda k: rewards[k])
        picked_reward.append(rewards[best])
        picked_truth.append(truths[best])
        oracle.append(max(truths[:n]))
        beats.append(int(rewards[best] > reference_reward))
    curve[n] = (sum(picked_reward)/len(scored), sum(picked_truth)/len(scored),
                sum(oracle)/len(scored), sum(beats)/len(scored))
    kl = math.log(n) - (n - 1) / n
    print(f'{n:>3}{curve[n][0]:>13.4f}{curve[n][1]:>11.4f}{curve[n][2]:>9.4f}'
          f'{curve[n][3]:>19.2f}{kl:>10.4f}')

In [ ]:
# Читаємо таблицю числами, а не на око.
reward_gain = curve[16][0] - curve[1][0]
truth_gain = curve[16][1] - curve[1][1]
oracle_gain = curve[16][2] - curve[1][2]

print(f'від n=1 до n=16 винагорода зросла на {reward_gain:+.4f}')
print(f'                справжня метрика — на {truth_gain:+.4f}')
print(f'                оракул            — на {oracle_gain:+.4f}')
print()
print(f'частка виграшу оракула, яку забрала винагорода: '
      f'{truth_gain/oracle_gain if oracle_gain else float("nan"):.1%}')
print()
print('Оракул показує, скільки в цих 16 кандидатах було приховано користі.')
print('Різниця між ним і тим, що вибрала винагорода, і є ціною проксі.')

## 16 · Перевіримо формулу KL прямим підрахунком

У лекції стверджується, що відхід best-of-n від початкової політики дорівнює
`ln n − (n−1)/n` — і не залежить ні від моделі, ні від винагороди, лише від `n`.

Це можна перевірити, не знаючи інтегралів. Вишикуємо всі можливі продовження
за винагородою й поділимо їх на `M` однакових за ймовірністю сходинок.
Найкращий із `n` кандидатів опиниться на сходинці `k` або нижче тоді й лише
тоді, коли **всі** `n` туди потрапили, — тобто з імовірністю `(k/M)^n`.
Далі складаємо KL за означенням.

In [ ]:
def kl_by_steps(n, M=2000):
    """KL прямою сумою по M сходинках черги за винагородою."""
    total = 0.0
    for k in range(1, M + 1):
        q = (k / M) ** n - ((k - 1) / M) ** n      # ймовірність попасти на сходинку k
        if q > 0:
            total += q * math.log(q * M)           # ln(нова / стара), стара = 1/M
    return total

print(f'{"n":>3}{"формула":>12}{"пряма сума":>13}{"різниця":>12}')
for n in (1, 2, 4, 8, 16, 32):
    closed = math.log(n) - (n - 1) / n
    direct = kl_by_steps(n)
    print(f'{n:>3}{closed:>12.4f}{direct:>13.4f}{abs(closed-direct):>12.6f}')
    assert abs(closed - direct) < 1e-3, f'формула розійшлась із сумою при n={n}'
print('\n✅ формула збігається з прямим підрахунком')

## 17 · Що з цього виходить

Дві половини зошита дали два різні уроки, і обидва варті того, щоб їх
сформулювати вголос.

**Перша.** Донавчання на суміші завдань не дає моделі абстрактної слухняності.
Воно вчить її формату відповіді. Завдання, чия відповідь має **нову форму**,
не переноситься зовсім — і без обмеженого вибору цього навіть не видно,
бо вільний argmax дає рівно нуль із суто технічної причини.

**Друга.** Модель винагороди — замінник, а не істина. Перебір кандидатів
підіймає винагороду надійно, а справжню метрику — лише доти, доки замінник
не помиляється. Оракул показує, скільки користі було в тих самих кандидатах:
різниця між оракулом і тим, що обрала винагорода, і є ціною замінника.

І наскрізний урок обох половин: **число без рубежу поруч — не результат**.
Ми порахували рубежі першими, до будь-якої моделі, і саме вони визначили,
що вважати успіхом.

In [ ]:
print(f'усього процесорного часу: {time.process_time() - STARTED:.0f} с')

## Завдання

### 🟢 Рівень 1
Додай четверте завдання — **«перше слово»**: вказівка «назви перше слово цього
рядка», вхід — рядок без першого слова, відповідь — саме́ це слово. Множина
допустимих відповідей у нього та сама, що в «далі», тобто звичайні слова.
Порахуй для нього три рубежі й нульовий постріл.

**Зроблено, якщо:** таблиця з пʼятьма числами (найчастіша · випадкова · стеля ·
нульовий постріл вільно · обмежено) і речення про те, з якого боку від більшого
рубежу стоїть нульовий постріл.

### 🟡 Рівень 2
Знайди, скільки пар потрібно моделі винагороди. Прожени навчання на 100 · 300 ·
600 парах при **однаковій** кількості кроків і заміряй згоду на невидимих парах.

**Зроблено, якщо:** є три числа згоди з рубежем 0.5 поруч і відповідь, з якої
кількості пар згода впевнено відривається від монетки.

### 🔴 Рівень 3
Заміни справжню метрику. Замість перекриття токенів візьми **точний збіг
першого токена** продовження й перебудуй криву best-of-n.

**Зроблено, якщо:** дві криві на одному графіку й відповідь на питання, чи
змінилось `n`, на якому справжня метрика перестає рости. Якщо не змінилось —
це теж відповідь, і її треба назвати.